# Chapter 1 · Notebook 1 of 3 — Introducing Token Factory

**AI Agent Course** · Nebius Token Factory × NVIDIA

Over the three notebooks of this chapter, we build **our first controlled agentic loop**: a model that can request an action, have our Python code execute it, and use the result. This notebook covers the first two steps of that journey: **choosing a model** and **calling it from Python**. Notebook 2 adds the tool loop and Notebook 3 checks whether the loop is fast enough and cheap enough to be usable.

**Prerequisites:** Python 3.10+, basic familiarity with REST APIs. No GPU, no prior LLM experience assumed.

**By the end of this notebook you can:** navigate the Token Factory catalog, read a model card and pick a model for a job, and make chat completion calls from Python — single-turn, styled with system prompts, and multi-turn.

## 1. What is Token Factory?

Open-sourced models are those in which all ingredients - including the weights, documentation, and often research - is publically available. Open source models are particularly powerful because they can be ran on your own servers.

In practice, that takes serious capacity and skills - GPU clusters, serving stacks, scaling, monitoring - and most teams have better things to do. **Managed inference** providers like Nebius Token Factory close that gap: they deploy and operate the open-source models for you. You can interact with these cheaper (and often equally as powerful!) models all in the playground or with simple API calls.

**In this course, we use Token Factory's public models which available to all users and billed per token which makes monitoring cost simple.**

## 2. Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Now we store that key somewhere our code can find it. We use a `.env` file: a plain text file that holds keys as `NAME=value` lines.
**Create it:**

1. In the same folder as this notebook, create a folder called `lesson` (if it doesn't exist yet).
2. Inside it, create a new plain text file named exactly `.env` (note: no name before the dot!). In VS Code or Jupyter: right-click the `lesson` folder, New File, type `.env`. From a terminal: `touch lesson/.env`.
3. Open it and paste in your key:

```
NEBIUS_API_KEY=paste-your-key-here
```

4. Save. That's it. No quotes, no spaces around the `=`.

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

One common gotcha: your operating system may hide files starting with a dot. If the file seems to vanish after you create it, it's still there. Your code will find it even if Finder or Explorer doesn't show it!

In [1]:
%pip install -q openai python-dotenv rich sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
# from google.colab import userdata
# os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

Keys loaded.

## 3. The model catalog

The model catalog is the home for all models on Token Factory. Here, you can view basic information, and types of availability.

<details>
<summary>Show screenshot: Model catalog</summary>

![Model catalog](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-catalog.png)

</details>

The catalog hosts frontier open models: **GLM 5.2**, **Kimi 3** and the **NVIDIA Nemotron family**, among others.

## 4. The NVIDIA Nemotron ecosystem

Our flagship models for this course are the **NVIDIA Nemotron** series, a family of open-source models from NVIDIA's. The Nemotron series is NVIDIA's latest and most impressive models - spanning both small and efficient to large and frontier-performing.  The family shares training lineage and behavior, so you can prototype on a small one and scale up without rewriting anything. The three we care about:

| | **Nemotron 3 Nano** | **Nemotron 3 Super** | **Nemotron 3 Ultra** |
|---|---|---|---|
| Routing key | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B` | `nvidia/nemotron-3-super-120b-a12b` | `nvidia/Nemotron-3-Ultra-550b-a55b` |
| Size | Smallest | Medium | Largest |
| Rough character | Fast and very cheap; great for everyday tasks, classification, drafting | The middle ground; strong general model for most workloads | The heavyweight; deepest reasoning, highest price |

A useful mental model: **start small, scale only when the task needs it.** The bigger the model, the more it costs per token and the slower it generates - and for a surprising share of real work, the small one is simply good enough!

**Our workhorse for this chapter is Nemotron 3 Nano.** We'll bring in its bigger siblings when we want a comparison, which will mimmic how you should scale with a real application.

## 5. Reading a model card

If you were deciding on an engine to be put in a car, you would likely read through different spec sheets until you found the one to fit your needs.

Every model in the catalog has a card - this is its spec sheet. The model cards are something you will come back to constantly: for the model string, for prices, for context limits, for what the model can and can't do. Need a fast car? You will pick the strongest - and likely most expensive - option. If you care about efficiency, you would choose one that is smaller and cheaper. The same choices will persist for the models you choose.

Click into **NVIDIA-Nemotron-3-Nano-30B-A3B**, scroll down, and click the model card.

<details>
<summary>Show screenshot: Nemotron 3 Nano model card</summary>

![Nemotron 3 Nano model card](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card.png)

</details>

### Copying the model string

At the top of the endpoint properties you'll find the **routing key** - for our model it's `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`, with a copy icon right next to it. **That exact string is what you pass as `model=` in every API call.** Always copy it from the card rather than typing it; a typo here is the single most common first error, and the API's "model not found" message won't tell you which character you got wrong.

### Reading the model card - what specs are important to us?

**Context window 262k.**
Find this number on the Nano card — it's the ceiling on everything the model can see at once: your prompt, the conversation history, any documents you paste in, *and* the output it generates. They all share this budget. Think of it as the model's working memory. A bigger window means you can hand it an entire codebase or a stack of contracts in one go; a smaller one means you'll have to chop things up and manage what the model gets to see.

**Pricing.**
Pricing is in terms of tokens, and there is a reason for the separation. Input tokens are cheap because the model ingests your whole prompt in a single go; output tokens cost more because the model generates them one at a time. Each one is a full trip through the model. Practical consequence: a chatty model that writes long answers costs you much more (think, what can we do to outputs to reduce this?). For scale: Ultra charges \$1.00 / \$3.00 for the same million tokens — roughly **15× more**.

**Modality: Text-to-text.**
What goes in and what comes out. Our model reads text and writes text, but scroll through the catalog and you'll see vision models (they accept images) and embedding models (they output vectors, not sentences). Checking this field takes two seconds and saves you from building half a project around a model that can't do what you assumed.

**Tool calling / Reasoning: Available.**
We'll return to what these mean later in the course — for now, just note that they're on the card, and that they'll matter a lot once we build agents.

**License.**
Open-weight does not mean do-whatever-you-want. Licenses differ on commercial use, redistribution, and what you can train on the outputs. Important to understand before putting a model in production, or starting work on something you may not be allowed to do.

### Exercise: pick a model by reading its card

Open two cards side by side: **`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`** and **`nvidia/Nemotron-3-Ultra-550b-a55b`**.

Two jobs. Tor each, decide which model you'd pick, and name the **one or two card fields** that settle it. One sentence of justification each ("it's bigger" doesn't count!):

1. **An everyday assistant** — rephrases emails, drafts short summaries, answers quick questions all day long.
2. **A contract analyst** — answers questions about 300-page legal documents pasted in whole, where a wrong answer is very costly.

*Hint: for one of these, the price gap is important. For the other, look at the context window and ask what a mistake costs.*

In [3]:
# Read the two cards from code: the catalog API returns the same fields the card page shows
from openai import OpenAI

BASE_URL = "https://api.tokenfactory.us-central1.nebius.com/v1/"
NANO = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B"
ULTRA = "nvidia/Nemotron-3-Ultra-550b-a55b"

catalog = OpenAI(base_url=BASE_URL, api_key=os.environ.get("NEBIUS_API_KEY"))
cards = {m.id: m.model_extra for m in catalog.models.list(extra_query={"verbose": "true"})}

for model_id in [NANO, ULTRA]:
    card = cards[model_id]
    price_in = float(card["pricing"]["prompt"]) * 1e6
    price_out = float(card["pricing"]["completion"]) * 1e6
    print(f"[bold]{model_id}[/bold]")
    print(f"  context window: {card['context_length']:,} tokens")
    print(f"  price per 1M:   ${price_in:.2f} input / ${price_out:.2f} output")
    print(f"  modality:       {card['architecture']['modality']} | features: {card['supported_features']}")

# Contract analyst: 300 pages at ~400 tokens/page (the rule of thumb from the video)
contract_tokens = 300 * 400
print(f"\n[bold]A 300-page contract is about {contract_tokens:,} tokens[/bold]")
for model_id in [NANO, ULTRA]:
    card = cards[model_id]
    share = contract_tokens / card["context_length"]
    cost = contract_tokens * float(card["pricing"]["prompt"])
    print(f"  {model_id}: fills {share:.0%} of the window, ${cost:.4f} input cost per question")

nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B

context window: 262,144 tokens

price per 1M:   $0.06 input / $0.24 output

modality:       text->text | features: ['tools', 'reasoning']

nvidia/Nemotron-3-Ultra-550b-a55b

...

price per 1M:   $1.00 input / $3.00 output

modality:       text->text | features: ['tools', 'reasoning']

A 300-page contract is about 120,000 tokens

nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B: fills 46% of the window, $0.0072 input cost per question

nvidia/Nemotron-3-Ultra-550b-a55b: fills 11% of the window, $0.1200 input cost per question

#### My answers

| Job | Pick | Card field(s) that settle it |
|---|---|---|
| **Everyday assistant** | **Nemotron 3 Nano** | **Pricing.** Nano costs \$0.06 / \$0.24 per 1M tokens versus Ultra's \$1.00 / \$3.00, so Ultra is about 17× more for input and 12.5× more for output. That's paid on every call, all day, for tasks Nano already handles well. The context window doesn't matter here: an email is a few hundred tokens. |
| **Contract analyst** | **Nemotron 3 Ultra** | **Context window, then pricing weighed against the cost of a mistake.** A 300-page contract is about 120k tokens. That's nearly half of Nano's 262k window before any conversation history or output, so follow-up questions and a second contract would overflow it. In Ultra's 1M window the same contract fills 11%. The pricing field shows Ultra's premium is about \$0.12 per question, which is trivial next to the cost of a wrong legal answer. |

## 6. First calls in the UI (Playground)

Before any code, use the **Token Factory playground**:

1. Open the model card for **Nemotron 3 Nano** and click **Go to playground**.
2. Send this prompt:

   > *A survey shows 62% of respondents prefer Product A and 38% prefer Product B, but the group that prefers A is three times smaller in the overall population. Which product actually wins overall? Show your reasoning briefly.*

3. Look at the **metrics below the response** - response time, token counts (it is OK to not fully understand these yet!).
<details>
<summary>Show screenshot: Nemotron response metrics</summary>

![Nemotron response metrics](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/request-metrics.png)

</details>

4. Now run the **exact same prompt** on **Nemotron 3 Ultra** side by side.

**What to compare:** the number of **output tokens** each model spent, how long each took, and the answers themselves - does the big model reason more carefully, or just write more? Did the small one get the right answer anyway? This is your first *evaluation*, and the honest answer to "which model is better" always starts with "for what purpose?".

#### Doing the comparison in code

Under each answer the playground shows three speed numbers: **ms** (time to first token), **s** (total time) and **T/s** (output tokens per second). Streaming the response gives the same numbers from code, so here's the Nano vs. Ultra comparison as a repeatable cell. Each model runs 3 times, because a single run is too noisy to compare speed.

In [4]:
import time
from statistics import median

SURVEY_PROMPT = (
    "A survey shows 62% of respondents prefer Product A and 38% prefer Product B, "
    "but the group that prefers A is three times smaller in the overall population. "
    "Which product actually wins overall? Show your reasoning briefly."
)


def timed_call(model_id):
    """Stream one request and collect the playground's metrics: time to first token, total time, tokens/s."""
    start = time.perf_counter()
    first_token = None
    answer, reasoning, usage = "", "", None
    stream = catalog.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": SURVEY_PROMPT}],
        stream=True,
        stream_options={"include_usage": True},  # final chunk carries the token counts
    )
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        piece = (delta.model_extra or {}).get("reasoning") or ""
        if (delta.content or piece) and first_token is None:
            first_token = time.perf_counter() - start
        answer += delta.content or ""
        reasoning += piece
    total = time.perf_counter() - start
    card = cards[model_id]
    cost = usage.prompt_tokens * float(card["pricing"]["prompt"]) + usage.completion_tokens * float(card["pricing"]["completion"])
    return {
        "ttft_ms": first_token * 1000,
        "total_s": total,
        # Over the whole request: Ultra reasons without streaming it, so its visible answer arrives in one burst
        # and dividing by (total - first_token) would give an absurd rate
        "tok_per_s": usage.completion_tokens / total,
        "out_tokens": usage.completion_tokens,
        "reasoning_chars": len(reasoning),
        "cost": cost,
        "answer": answer.strip(),
    }


for model_id in [NANO, ULTRA]:
    runs = [timed_call(model_id) for _ in range(3)]
    print(f"[bold]--- {model_id} ---[/bold]")
    for r in runs:
        print(f"  {r['ttft_ms']:5.0f} ms | {r['total_s']:4.1f} s | {r['tok_per_s']:4.0f} T/s | "
              f"{r['out_tokens']:4} output tokens | reasoning {r['reasoning_chars']:,} chars | ${r['cost']:.6f}")
    print(f"  median: {median(r['ttft_ms'] for r in runs):.0f} ms | {median(r['total_s'] for r in runs):.1f} s | "
          f"{median(r['tok_per_s'] for r in runs):.0f} T/s | {median(r['out_tokens'] for r in runs):.0f} output tokens")
    print(runs[0]["answer"], "\n")

--- nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B ---

1212 ms |  3.8 s |  228 T/s |  861 output tokens | reasoning 2,674 chars | $0.000210

2095 ms |  4.3 s |  177 T/s |  754 output tokens | reasoning 2,011 chars | $0.000185

1765 ms | 36.6 s |   37 T/s | 1344 output tokens | reasoning 4,160 chars | $0.000326

median: 1765 ms | 4.3 s | 177 T/s | 861 output tokens

...

10295 ms | 10.5 s |  324 T/s | 3397 output tokens | reasoning 0 chars | $0.010254

16962 ms | 17.0 s |  217 T/s | 3687 output tokens | reasoning 0 chars | $0.011124

3659 ms |  4.0 s |  235 T/s |  936 output tokens | reasoning 0 chars | $0.002871

median: 10295 ms | 10.5 s | 235 T/s | 3397 output tokens

Product B wins overall.

**Reasoning:** The statement "the group that prefers A is three times smaller in the overall population" means the 
number of people preferring A is one-third the number preferring B. Thus, B has three times as many supporters as A
in the overall population, regardless of the survey's 62/38 split among respondents.

#### My notes on the comparison

The correct answer is **Product B**. If A-fans are a third the size of B-fans in the population, A has 25% of the market and B has 75%; the 62/38 survey split only reflects who answered. Both models' printed answers get this right.

Medians of 3 runs each, from the cell above:

| | Nano | Ultra |
|---|---|---|
| Time to first token (playground "ms") | ~1.8 s | ~10.3 s |
| Total time (playground "s") | ~4.3 s | ~10.5 s |
| Tokens per second (playground "T/s") | ~180 | ~235 |
| Output tokens | ~860 | ~3,400 (range 940–3,700) |
| Cost per answer | ~\$0.0002 | ~\$0.010 |

- **Did the bigger model think more?** Yes, much more. Ultra used about 4× the output tokens, almost all of it hidden reasoning; its visible answer is only a couple hundred tokens.
- **Did the bigger model write a better answer?** Not on this prompt. Both get B at 75%, and Ultra's write-up is only slightly tighter.
- **Did it take longer?** Yes. Ultra returns no `reasoning` field, so nothing shows until it finishes thinking: about 10 s before the first visible token, versus under 2 s for Nano, whose reasoning streams.
- **The real cost gap is bigger than the price gap.** Ultra's output price is 12.5× Nano's, but because it also thinks about 4× longer, each answer cost about **50× more**. Compare models on cost per answer, not just price per token.
- **Single runs are noisy.** One Nano run took 36.6 s because of server queueing, and Ultra's token count varied 4× across runs. That's why the cell runs each model 3 times and reports medians.

## 7. The Chat Completions request

The Playground isn't for real applications, it's for *you* to evaluate a model. To use a model in real workflows, you need to call it from code, and the Token Factory API is exactly for that.

A Chat Completions request is the standard way to interact with models via API. It is a structured format for how we communicate to models and how they communicate back. A sample completions request for your model can be seen on the model card:

<details>
<summary>Show screenshot: Chat Completions example</summary>

![Chat Completions example](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card-completions.png)

</details>

### Token Factory's API is OpenAI-compatible

If you ever went through any OpenAI API tutorial, this will be easy-peasy for you: everything works the same way, you just change the `base_url` and the model names. That compatibility is a big deal in practice:

- The entire ecosystem of tools works out of the box
- Drop-in migration from other providers
- Every tutorial on the internet works with a **one-line `base_url` change**

### 7.1 Anatomy of the request

Before the code, the pieces:

- **`model`** *(required)* — the routing key you copied from the card.
- **`messages`** *(required)* — the conversation as a list of roles. **Key note: the API is stateless — you send the whole conversation history every time.** The roles:
  - **`system`** — instructions for how the model should behave. Less about giving it expertise, more about **tone of voice and style**: the same question answered by "a formal analyst" and "a casual intern" reads completely differently. We'll see this live in a minute. Set once, at the top.
  - **`user`** — what the human says. Your prompts go here!
  - **`assistant`** — what the model said. You append its previous replies here so it remembers the conversation.
  - **`tool`** — results from function calls, which we'll meet in the next notebook.
- **Sampling params** *(optional)* — `temperature`, `top_p`, `max_tokens`. We will not worry about these for now.
- **The response object** — what comes back. We'll open one up right after the first call.

### 7.2 Minimal call

The model card provides a ready-made snippet ("Use model in code" - the one below is adapted from it), so you never have to write this boilerplate from memory:

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

NVIDIA’s Nemotron model family is designed to accelerate the training and inference of large‑scale generative AI 
workloads for enterprise and research applications, offering specialized optimizations for both latency‑sensitive 
and throughput‑intensive scenarios. By integrating tightly with NVIDIA’s hardware and software stack, it enables 
faster model deployment, reduced compute costs, and scalable performance across cloud, on‑premise, and edge 
environments.

### 7.3 What came back? A one-time tour of the response

The response isn't just text - it's a structured object, and a few of its fields matter for everything we do later. Let's dump the whole thing **once**, look around, and then never print it in full again:

In [6]:
# One-time tour — after this cell, we only ever print .content
print(response.to_json())

# The fields worth remembering:
print("---")
print("content:       ", response.choices[0].message.content[:60], "...")   # the answer
print("finish_reason: ", response.choices[0].finish_reason)                  # why it stopped ("stop" = natural end, "length" = hit max_tokens)
print("usage:         ", response.usage)                                     # tokens in/out — this becomes our COST signal in Notebook 3

{
  "id": "chatcmpl-1351c868-e287-4708-b2a6-e9c9420fc9ca",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "\nNVIDIA’s Nemotron model family is designed to accelerate the training and inference of 
large‑scale generative AI workloads for enterprise and research applications, offering specialized optimizations 
for both latency‑sensitive and throughput‑intensive scenarios. By integrating tightly with NVIDIA’s hardware and 
software stack, it enables faster model deployment, reduced compute costs, and scalable performance across cloud, 
on‑premise, and edge environments.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": "The user asks: \"In two sentences, what does NVIDIA's Nemotron model family target?\" So 
response should be two sentences, concise, explain target of Nemotron model family. Provide answer.\n"
      },
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "created": 1792143474,
  "model": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
  "object": "chat.completion",
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": "vllm-0.22.0-b08c96cd",
  "usage": {
    "completion_tokens": 130,
    "prompt_tokens": 44,
    "total_tokens": 174,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  },
  "prompt_logprobs": null,
  "prompt_token_ids": null,
  "prompt_text": null,
  "kv_transfer_params": null
}

---

content:        
NVIDIA’s Nemotron model family is designed to accelerate th ...

finish_reason:  stop

usage:         
CompletionUsage(
    completion_tokens=130,
    prompt_tokens=44,
    total_tokens=174,
    completion_tokens_details=None,
    prompt_tokens_details=None
)

Three things to file away: **`content`** is the answer, **`finish_reason`** tells you *why* generation stopped (watch for `"length"` - it means the answer was cut off), and **`usage`** counts prompt and completion tokens - the number your bill is computed from. From here on, our cells print only the content, for readability.

### 7.4 System prompts: same question, two voices

The claim above was that system prompts are mostly about tone and style. Prove it! Same user question, two different personas:

In [7]:
QUESTION = "Is the specialty coffee market growing?"

for persona in [
    "You are a formal senior market analyst. Precise, measured, no exclamation points.",
    "You are an enthusiastic intern on the market-research team. Casual, punchy, emoji welcome.",
]:
    resp = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": QUESTION},
        ],
    )
    print(f"[bold]--- {persona[:40]}... ---[/bold]")
    print(resp.choices[0].message.content, "\n")

--- You are a formal senior market analyst. ... ---

La industria del café especializado muestra una expansión sostenida en los últimos años. Según los últimos informes
de Euromonitor y Specialty Coffee Association, el valor global del segmento ha registrado un crecimiento anual 
promedio de entre 8 % y 10 % desde 2018, impulsado por la mayor demanda de calidad y experiencias diferenciadas. En
EE. UU., el consumo per cápita de café de origen y preparaciones artesanales ha pasado de 2,5 kg en 2015 a casi 3,3
kg en 2023, reflejando una penetración más profunda de estos productos en el hábitos cotidianos.

En el contexto de los consumidores latinos, la tendencia se refuerza por la convergencia de dos fuerzas: la 
aspiración a experiencias premium y la incorporación de rituales de consumo que se alinean con la cultura del café 
artesanal. Las plataformas digitales facilitan la educación del cliente mediante contenidos en japonés, inglés y 
ruso, lo que permite ampliar la base de compradores internacionales y crear segmentos de nicho más rentables.

Los datos confirman que la categoría sigue en expansión, aunque la tasa de crecimiento se modula por la capacidad 
de los actores a ofrecer diferenciación auténtica y narrativas de origen coherentes. Nuestra experiencia colectiva 
indica que la combinación de storytelling culturalmente rico y canales digitales bien gestionados es el motor que 
sostiene dicha expansión.

--- You are an enthusiastic intern on the ma... ---

Hey there! 🙌  

**Short answer:** *Absolutely*—the specialty coffee market is on a serious up‑trend.  

**A few quick stats I’ve been digging into:**  

- 📈 The global specialty coffee market is projected to hit **$85 B by 2027**, growing at **~12% CAGR** from 
2022‑2027 (Grand View Research).  
- 🏊‍♀️ In the U.S., specialty‑coffee sales (third‑wave cafés, premium beans, subscription boxes) jumped **~20% YoY**
in 2023—still outpacing the broader coffee industry (~8% growth).  
- 🌍 Consumer willingness to pay more for quality is driving the surge: **70% of coffee drinkers** say they’ll 
spend extra for “better flavor,” ethically sourced beans, or unique brewing methods.  
- 🚀 Emerging hubs like **Scandinavia, East Asia, and the Middle East** are adding double‑digit growth rates—so 
it’s not just a U.S. thing anymore.  

**Why it matters to us:**  
- More retailers and cafés are *curating* specialty lines → more shelf‑space and partnerships for us.  
- **Subscription‑box demand** is exploding (think “coffee of the month” clubs), giving us fresh avenues for product
placement.  
- The **sustainability & traceability** buzz means brands are looking for authentic stories—perfect for our 
content‑driven campaigns.  

So, yep—specialty coffee isn’t just staying hot; it’s brewing faster and hotter than ever! ☕🔥 Let’s ride this 
wave together. 🌊🚀  

(Feel free to ping me if you want deeper dive reports or visual slides!)

Same facts, completely different voice. In an agent, the system prompt is where you pin down *how* it should behave - style, format, boundaries, etc.

### 7.5 Multi-turn: the API is stateless, so *you* carry the memory

Ask a follow-up question in a fresh request and the model has no idea what you're talking about - nothing is stored server-side. "Memory" is literally you resending the conversation. First, fully by hand, so you see every move:

In [8]:
# Turn 1: ask
history = [
    {"role": "system", "content": "You are a concise assistant for a market-research team."},
    {"role": "user", "content": "Name three fast-growing beverage categories."},
]
resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
answer_1 = resp.choices[0].message.content
print(answer_1)

# Turn 2: append the model's answer, then our follow-up, then resend EVERYTHING
history.append({"role": "assistant", "content": answer_1})
history.append({"role": "user", "content": "Which of those is cheapest to enter as a small brand?"})

resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
print("---")
print(resp.choices[0].message.content)   # it knows "those" only because we resent the history

**Fast‑growing beverage categories (2023‑2024)**  

1. **Non‑alcoholic functional drinks** – e.g., ready‑to‑drink adaptogenic teas, sparkling water‑based nootropics, 
and probiotic sodas.  
2. **Low‑/no‑sugar ready‑to‑drink (RTD) teas & coffees** – e.g., canned matcha, cold‑brew coffee, and tea‑infused 
sparkling beverages.  
3. **Plant‑based dairy alternatives** – e.g., oat‑milk lattes, almond‑milk cold brews, and coconut‑milk lattes that
are marketed as “clean‑label” milk substitutes.  

These categories have shown double‑digit CAGR over the past 12‑month period, driven by consumer demand for 
health‑focused, low‑calorie, and sustainably packaged options.

---

### Quick Verdict  
**The cheapest category to break into as a small, boot‑strapped brand is **`Low‑/no‑sugar Ready‑to‑Drink (RTD) teas
& coffees`**.  

Below is a concise why‑it‑makes‑sense breakdown, plus a few practical tips to keep the entry cost under $30 K (or 
even under $10 K if you’re ultra‑lean).

---

## 1. Cost Drivers Compared

| Factor | Non‑alcoholic functional drinks | Low‑/no‑sugar RTD teas & coffees | Plant‑based dairy alternatives |
|--------|--------------------------------|-----------------------------------|--------------------------------|
| **Formulation complexity** | Often requires multiple actives (adaptogens, nootropics, probiotics) → R&D, 
stability testing, ingredient sourcing (higher per‑kg cost) | Simple base (tea or coffee) + optional natural 
flavor/ sweetener + carbonation or cold‑brew process → many ingredients are commodity‑grade | Requires emulsifiers,
stabilizers, pH control, and often additional vitamins/minerals → more R&D, stricter labeling, and higher 
ingredient spend |
| **Regulatory pathway** | “Functional” claims trigger FDA/NUHDS (U.S.) or EFSA (EU) scrutiny; must substantiate 
health claims → extra testing & documentation | Usually falls under “beverage” category only; modest labeling (no 
health claims) → easier compliance | Often classified as “food” with fortified claims; may need nutrition facts, 
allergen statements, and possibly GRAS notices for novel ingredients → higher legal/CPA costs |
| **Packaging** | Typically cans or bottles (requires BPA‑free or specialty containers) → higher unit costs | Cans 
or PET bottles are standard; many co‑packers already serve RTD tea/coffee → lower MOQs | Often needs cartons with 
secondary packaging that meets dairy‑like shelf life (refrigeration or aseptic) → higher packaging spend |
| **Manufacturing scale‑up** | Needs specialized equipment (e.g., bottling lines with nitrogen flush for 
probiotics) → niche contract manufacturers can be pricey | Standard beverage bottling lines; many small‑batch 
co‑packers exist, low minimum order quantities (as few as 5 K units) | High‑volume aseptic processing or 
refrigerated co‑packing → higher per‑unit fees for low volumes |
| **Marketing entry cost** | Brand stories around “functional benefits” → need scientific content, claims 
substantiation → more content spend | Story can focus on taste, low‑sugar, “clean label” → lighter content budget; 
can lean on influencer tea/coffee communities | Must educate consumers on plant‑based vs dairy, often higher CPM 
for educational ads |
| **Overall startup capex (ballpark)** | $30 K‑$80 K (incl. formulation, label compliance, small batch) | $10 K‑$30
K (incl. label design, co‑packer, small batch) | $40 K‑$120 K (incl. formulation, lab testing, packaging, 
refrigeration logistics) |

> **Takeaway:** The functional‑drink niche and plant‑based dairy have higher R&D, regulatory, and packaging 
overheads, pushing the breakeven point higher. RTD teas/coffees sit at the sweet spot of low ingredient complexity,
minimal health‑claim burden, and a mature co‑packing ecosystem.

---

## 2. Why RTD Teas & Coffees Are the Lowest‑Barrier Entry Point

| Reason | Detail |
|--------|--------|
| **Ingredient simplicity** | Black/green tea, ready‑to‑drink coffee concentrates, natural flavor extracts, and 
non‑caloric sweeteners (stevia, monk fruit) are commodity‑grade and bought in bulk for <$1 /kg. |
| **No functional claim burden** | You can market as “unsweetened”, “zero‑sugar”, “caffeinated”, “caffeine‑free”, 
or “clean label” without needing FDA health‑claim approvals. |
| **Mature co‑packing ecosystem** | Many small co‑packers specialize in cold‑brew, bottled iced teas, or 
ready‑to‑drink coffee drinks. They accept MOQs as low as 5 K‑10 K units and can handle custom labeling. |
| **Flexible packaging** | Both cans (e.g., 12 oz) and PET bottles (16 oz) are interchangeable; you only need a 
single packaging spec to start. |
| **Short shelf‑life pathway** | Non‑refrigerate

That works, but the appending pattern is always the same, so wrap it in a tiny loop and you have a chat:

In [9]:
history = [{"role": "system", "content": "You are a concise assistant for a market-research team."}]

def ask(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    return answer

print(ask("Name three fast-growing beverage categories."))
print("---")
print(ask("Which of those is cheapest to enter as a small brand?"))

**Three fast‑growing beverage categories**

1. **Hard seltzer / low‑ABV ready‑to‑drink (RTD) cocktails** – high‑growth in the U.S. and Europe, driven by demand
for low‑calorie, portable options.  
2. **Plant‑based and alternative‑milk “beverages”** (e.g., oat, almond, soy, pea) – expanding beyond coffee to 
everyday consumption and food‑service use.  
3. **Functional & adaptogenic drinks** (e.g., sparkling water with added nootropics, botanicals, vitamins) – rapid 
adoption as consumers seek health‑enhancing, low‑sugar refreshment.

---

**The cheapest of the three to break into as a small, niche brand is **_Functional & adaptogenic drinks (e.g., 
sparkling water infused with herbs, nootropics, vitamins, or botanicals)_**.

### Why it’s the lowest‑cost entry point

| Factor | Functional/Adaptogenic Drinks | Hard Seltzer | Plant‑Based “Milk‑Beverages” |
|--------|-------------------------------|--------------|------------------------------|
| **Ingredient base** | Simple carbonated water + low‑cost functional extracts (e.g., adaptogens, herbal infusions,
vitamins). Minimal processing, no heavy dairy or grain infrastructure. | Requires fermentable sugar, yeast, 
flavorings, and often a malt base; higher COGS and stricter regulatory (alcohol) compliance. | Needs a 
plant‑protein or nut base, emulsifiers, fortification, and often a longer shelf‑life process (UHT or high‑pressure 
processing). |
| **Packaging** | Standard 330 ml or 500 ml PET/ aluminum can or bottle; can start with contract bottling using 
existing label lines. | Similar packaging, but you must meet excise‑tax and labeling rules for alcoholic beverages;
usually requires larger volumes to be economical. | Shelf‑stable cartons or glass bottles with higher material 
cost; often need aseptic filling lines. |
| **Regulatory & licensing** | Mostly non‑alcoholic; only needs FDA GRAS status for ingredients, standard 
food‑label compliance, and possibly a dietary‑supplement claim if marketed that way. | Must obtain alcohol 
manufacturing permits, comply with state‑by‑state alcohol laws, and meet labeling for “malt‑based” or 
“distilled‑spirit” equivalents. | Must satisfy dairy‑alternatives regulations, often require nutrient profiling and
sometimes a “milk” label (which can trigger specific compositional standards). |
| **Manufacturing scale** | Can launch with a **private‑label contract manufacturer** that fills a few hundred to a
few thousand cases per run. Low minimum order quantities (MOQ) are common. | Contract brewing typically demands 
**larger batch sizes** (10,000+ cans) to make the economics work, and you’ll need to share a production slot with a
brewery. | Requires aseptic filling or high‑volume packaging lines; small‑run production is pricey and usually only
viable for larger CPG players. |
| **Marketing entry cost** | Story‑driven (wellness, lifestyle) can be communicated with minimal spend; influencer 
micro‑campaigns, social‑media teasers, and direct‑to‑consumer (DTC) subscription models work well. | Requires 
compliance with alcohol advertising rules; often needs higher media spend to build brand awareness. | Branding must
overcome entrenched dairy alternatives; advertising budgets are typically higher to differentiate on 
taste/nutrition. |
| **Supply‑chain simplicity** | Bulk carbonated water, a few dried extracts or powders, and a sweetener—sourced 
from commodity suppliers with stable pricing. | Requires fermentable sugars, yeast, and often a malt base, which 
can be price‑volatile and subject to seasonal supply constraints. | Sourcing nuts, legumes or oats, plus 
fortification minerals, adds complexity and often higher per‑unit cost. |

### Practical Steps to Launch a Low‑Cost Functional/Adaptogenic Drink

1. **Formulate a simple “sparkling functional water”** – carbonated water + 1‑2 g of a well‑studied adaptogen 
(e.g., ashwagandha, lion’s mane, rhodiola) + a natural sweetener (stevia/monk fruit) + a light flavor (citrus, 
ginger, berry).  
2. **Partner with a contract bottler** that already has a line for sparkling water; many will fill a few hundred 
cases for a flat fee.  
3. **Secure a single‑ingredient supplier** with bulk discounts (e.g., bulk ashwagandha extract from India).  
4. **Label compliance:**  
   - Keep the product **non‑alcoholic** and **non‑medicinal** (avoid “health claim” language unless you pursue a 
qualified health claim).  
   - Use a **structured “Supplement Facts”** panel only if you truly intend to market it as a dietary supplement; 
otherwise, a “Nutrition

Notice what the loop implies about cost: every turn resends the *whole* history, so long conversations re-pay for everything already said.

### Exercise: try to create a request that does these tasks!

All of these use our market-research assistant. Fill in the `## What goes here?` gaps.

#### 1. Only produce 10 tokens or less per response

In [10]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Give me a one-line headline on the energy-drink market."

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
    max_tokens=10,  # hard cap on generated tokens
    # Nano is a reasoning model: with thinking on, its hidden reasoning spends the 10 tokens and content comes back None
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)  # what do you expect to see here?

Energy‑drink sales are projected to hit $

---

finish_reason: length

**Answer:** `max_tokens=10`, and `finish_reason` is `"length"` because the headline gets cut off mid-sentence. A prompt like "answer in 10 tokens" isn't enough on its own; only `max_tokens` enforces the cap.

**Gotcha I hit:** with thinking left on, the 10 tokens all went to Nano's hidden reasoning (`message.reasoning`), and `content` came back as `None`. Turning thinking off with `chat_template_kwargs` gives the visible answer the whole budget. Reasoning tokens count toward `max_tokens` and toward `usage.completion_tokens`, so you pay for them too.

#### 2. Change the *voice*: make the assistant answer like a skeptical veteran analyst who distrusts hype

In [11]:
SYSTEM_PROMPT = (
    "You are a skeptical veteran market analyst with 30 years in the industry. "
    "You've watched countless 'booming' markets fizzle, so you distrust hype, press releases, and vendor-funded reports. "
    "Question the source and methodology behind every growth claim, separate real demand from marketing noise, "
    "and call out what the bulls are ignoring. Dry, blunt, and unimpressed. No exclamation points."
)
USER_MESSAGE = "Is the specialty coffee market growing?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

The numbers show specialty coffee is expanding, but only modestly—roughly 4‑5 % annual growth in U.S. retail sales 
over the past three years, according to the Specialty Coffee Association’s own point‑of‑sale data. International 
figures are less reliable; many of the cited “global” growth rates rely on franchise‑level revenue estimates that 
include both specialty and fast‑service coffee chains, blurring the line between boutique and mainstream.  

Demand exists, but it’s concentrated in urban, higher‑income demographics. The market is already dense in major 
metros, and the incremental growth largely comes from younger consumers swapping out conventional brews for 
“third‑wave” options rather than a broader shift across the general population. Supply constraints—limited domestic
bean production, price volatility of high‑altitude coffees, and the cost of establishing specialty supply 
chains—keep expansion capped.  

In short, the sector is growing because niche consumers are willing to pay a premium, not because it’s on the cusp 
of a mass‑market boom. The growth vector is narrow, and the upside is limited to those who can secure consistent, 
high‑quality sourcing and maintain differentiated positioning. That’s the bull case that most product‑centric 
reports gloss over.

#### 3. Challenge! Make the model produce 10 identical responses with:

```python
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."
```

**Hint: you may need to use a loop and a certain parameter that makes the model more deterministic (less creative)**

In [12]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."

responses = []
for _ in range(10):
    response = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_MESSAGE},
        ],
        temperature=0,  # greedy decoding: always take the most likely next token
        seed=42,        # fixed seed in case anything is still sampled
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # no hidden reasoning to drift
    )
    responses.append(response.choices[0].message.content)

print(response.choices[0].message.content)
print("---")
print(f"unique responses: {len(set(responses))} / {len(responses)}")

- **Smooth, low‑acid profile** appeals to health‑conscious and sensitive‑stomach consumers.  
- **Convenient ready‑to‑drink formats** (bottles, cans) fit busy lifestyles and on‑the‑go consumption.  
- **Premium perception** drives higher price points and willingness to pay for specialty cold brew.  
- **Versatile serving options** (milk, sweeteners, flavor infusions) attract diverse taste preferences.  
- **Strong social‑media buzz** fuels viral trends, encouraging trial and repeat purchases.

---

unique responses: 10 / 10

**What I found:** the intended answer is a loop plus `temperature=0` (with a fixed `seed`). In testing on Token Factory, though, the 10 responses still weren't byte-identical. The wording and bullet order drift between calls even with `temperature=0`, `seed`, `top_k=1` and thinking off, on both the regional and the global endpoint. Temperature itself is honored: at `temperature=2.0` the output turns into multilingual gibberish.

The likely cause is server-side non-determinism. Nano is a mixture-of-experts model served in shared batches, so tiny floating-point differences change which expert or token wins early, and the text diverges from there. **Takeaway:** `temperature=0` makes a hosted model *more* consistent, not identical. If you need the exact same output every time, cache the response; don't count on regenerating it.

## Wrap-up!

You created an account and an API key, and stored it in a dot-env file so it stays out of your code. You learned to read a model card - the routing key, the context window and the three things sharing it, the two pricing rates and why output costs more, modality, and the license. You tested a model in the playground and compared a small one against a large one on the same prompt. And then you built the same interaction in code: a single request, the response object with its content, finish reason, and usage, a system prompt to control voice, and a multi-turn conversation where you carry the history yourself.

Together, that means you can choose a model for a job, call it from Python, and hold a real conversation with it while knowing exactly what each request cost you.

You now have a working foundation - a model you've chosen deliberately, and the code to talk to it.
